In [1]:
#Quick test
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count

spark = SparkSession.builder.appName("DataprocTest").getOrCreate()
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

data = [
    (1, "Alice", "Engineering", 95000),
    (2, "Bob", "Marketing", 75000),
    (3, "Charlie", "Engineering", 105000),
    (4, "David", "Finance", 85000),
    (5, "Emma", "Marketing", 80000),
    (6, "Frank", "Finance", 90000),
]

df = spark.createDataFrame(data, ["id", "name", "department", "salary"])
df.show()
df.groupBy("department").agg(
    count("*").alias("employee_count"),
    avg("salary").alias("average_salary")
).show()
print("Total employees:", df.count())  # Expected: 6
print("Spark application ID:", spark.sparkContext.applicationId)

df.createOrReplaceTempView("employees")
spark.sql("""
    SELECT department, COUNT(*) AS employees, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""").show()

26/11/16 17:45:38 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Spark version: 3.5.3
Spark master: yarn


+---+-------+-----------+------+
| id|   name| department|salary|
+---+-------+-----------+------+
|  1|  Alice|Engineering| 95000|
|  2|    Bob|  Marketing| 75000|
|  3|Charlie|Engineering|105000|
|  4|  David|    Finance| 85000|
|  5|   Emma|  Marketing| 80000|
|  6|  Frank|    Finance| 90000|
+---+-------+-----------+------+



+-----------+--------------+--------------+
| department|employee_count|average_salary|
+-----------+--------------+--------------+
|Engineering|             2|      100000.0|
|  Marketing|             2|       77500.0|
|    Finance|             2|       87500.0|
+-----------+--------------+--------------+

Total employees: 6
Spark application ID: application_1794850938829_0001


+-----------+---------+----------+
| department|employees|avg_salary|
+-----------+---------+----------+
|Engineering|        2|  100000.0|
|    Finance|        2|   87500.0|
|  Marketing|        2|   77500.0|
+-----------+---------+----------+



In [2]:
#Run a distributed Spark job
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("DistributedSparkDemo").getOrCreate()

# Eight partitions provide multiple tasks for the cluster to schedule.
df = spark.range(0, 1_000_000, 1, 8)

print("Partitions:", df.rdd.getNumPartitions())

result = df.groupBy((col("id") % 10).alias("group")).count()
result.orderBy("group").show()

26/11/16 17:47:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Partitions: 8


+-----+------+
|group| count|
+-----+------+
|    0|100000|
|    1|100000|
|    2|100000|
|    3|100000|
|    4|100000|
|    5|100000|
|    6|100000|
|    7|100000|
|    8|100000|
|    9|100000|
+-----+------+



In [3]:
print("Spark master:", spark.sparkContext.master)
print("Spark application ID:", spark.sparkContext.applicationId)

Spark master: yarn
Spark application ID: application_1794850938829_0001
